# Otimização — capítulo 3: Otimização multidimensional sem restrições

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_5_hooke_jeeves/ex03_2_5_hooke_jeeves.ipynb)

Corra a célula **Preparação** e depois o exemplo (no Colab: *Runtime → Run all*). No fim, o exemplo compara os resultados com os slides (*confere com os slides: sim*). Os slides usam vírgula decimal; aqui usa-se o ponto. Caderno gerado a partir de `ex03_2_5_hooke_jeeves.py` por `notebooks/make_notebooks.py`.

*Code comments and printed messages are in Portuguese.*

## Preparação

In [1]:
# Preparação: põe as funções da UC no caminho do Python.
# No Colab (ou noutra pasta), clona o repositório; dentro do repositório, usa code/python.
import os, sys
LOCAL = [p for p in ("..", os.path.join("..", "..")) if os.path.isfile(os.path.join(p, "uc_setup.py"))]
if LOCAL:
    RAIZ = os.path.abspath(LOCAL[0])
else:
    if not os.path.isdir("optimization-course"):
        !git clone -q --depth 1 https://github.com/aguilarmadeira/optimization-course
    RAIZ = os.path.abspath(os.path.join("optimization-course", "code", "python"))
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import uc_setup  # acrescenta as pastas dos métodos e common/
print("Código da UC pronto.")

Código da UC pronto.


## 3.2.5 Método de Hooke–Jeeves

Exemplo [`ex03_2_5_hooke_jeeves.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_5_hooke_jeeves/ex03_2_5_hooke_jeeves.py) · funções da UC: [`exploratory.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_5_hooke_jeeves/exploratory.py), [`hooke_jeeves.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_5_hooke_jeeves/hooke_jeeves.py)

```text
Reproduz os exemplos do deck 3.2.5 (Hooke–Jeeves).

f(x) = 3 x1^2 + x2^2 - 12 x1 - 8 x2  (mínimo em (2,4), f = -28):
  «Uma exploração à mão»: de xb = (1,1), f = -16, com P = (0.5, 0.5):
    (1.5,1) -> -18.25, (1.5,1.5) -> -21, com 2 avaliações;
  «Exemplo completo»: x0 = (1,1), a = 2, P0 = (0.5,0.5), T = (0.1,0.1):
    padrão (2,2) -> (2,2.5), aceite; (2.5,3.5) -> (2,4), aceite;
    (2,5.5) -> (2,5), rejeitado; exploração em (2,4) falha com P = 0.5,
    0.25, 0.125; 0.0625 < T: para em (2,4), f = -28.
Rosenbrock a partir de (-1.5, 2), P0 = 0.5, a = 2, T = 1e-6 (como no
  caderno cap3_comparacao.ipynb): n_f = 353 até f < 1e-4.
Custo: n <= n_f <= 2n por exploração; 1 + (n a 2n) por movimento de padrão.

Contagens: f(x0) conta; f(xb) nunca é reavaliado (fica guardado).
Os slides usam vírgula decimal; aqui usa-se o ponto.
```

In [2]:
import numpy as np

from exploratory import exploratory
from hooke_jeeves import hooke_jeeves


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


ok = True
quad = lambda x: 3 * x[0]**2 + x[1]**2 - 12 * x[0] - 8 * x[1]
rosen = lambda x: (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

print("Otimização — deck 3.2.5: método de Hooke–Jeeves")

# --------------------------------------------- «Uma exploração à mão»
xb = np.array([1.0, 1.0]); fb = quad(xb); P = [0.5, 0.5]
print("\nExploração à mão: xb = (%g,%g), f(xb) = %g, P = (%g,%g)" % (*xb, fb, *P))
r = exploratory(quad, xb, fb, P)
for h in r.history:
    print("  x%d %s P%d: (%g, %g) -> f = %g, %s"
          % (h[0], "+" if h[1] > 0 else "-", h[0], h[2], h[3], h[4],
             "melhora: guarda-se" if h[5] else "não melhora"))
print("xe = (%g, %g), f = %g, com %d avaliações" % (*r.x, r.fx, r.nfev))
c = [confere(fb, -16, 0), confere(r.history[:, 2:5], [[1.5, 1, -18.25], [1.5, 1.5, -21]], 2),
     confere(r.x, [1.5, 1.5], 1), r.nfev == 2]
print("  f(1,1) = -16: %s | -18.25 e -21: %s | xe = (1.5,1.5): %s | 2 avaliações: %s" % simnao(c))
ok = ok and all(c)

# --------------------------------------------- «Exemplo completo»
x0, a, P0, T = [1, 1], 2, [0.5, 0.5], [0.1, 0.1]
print("\nExemplo completo: x0 = (%g,%g), a = %g, P0 = (%g,%g), T = (%g,%g)" % (*x0, a, *P0, *T))
r = hooke_jeeves(quad, x0, a, P0, T, verbose=True)
print(r.message)
print("solução x = (%g, %g), f = %g;  n_f = %d" % (*r.x, r.fx, r.nfev))
H = r.history
# linhas 2 a 4 do slide (padrão): xt(1) xt(2) f(xt)  xe'(1) xe'(2) f(xe')  aceite
Tpad = [[2.0, 2.0, -24.00, 2.0, 2.5, -25.75, 1],
        [2.5, 3.5, -27.00, 2.0, 4.0, -28.00, 1],
        [2.0, 5.5, -25.75, 2.0, 5.0, -27.00, 0]]
c = [r.nit == 7 and list(H[:, 1]) == [0, 1, 1, 1, 0, 0, 0],   # explor., 3 padrões, 3 explor.
     confere(H[0, 7:10], [1.5, 1.5, -21], 2),
     confere(H[1:4][:, [4, 5, 6, 7, 8, 9, 13]], Tpad, 2),
     confere(H[4:7, 11], [0.5, 0.25, 0.125], 4) and bool(np.all(H[4:7, 13] == 0)),
     confere(r.P, [0.0625, 0.0625], 4),
     confere(r.x, [2, 4], 4) and confere(r.fx, -28, 4)]
print("  sequência de movimentos: %s | 1.ª exploração -> (1.5,1.5): %s | padrões (2,2), (2.5,3.5), (2,5.5): %s |\n"
      "  falha com P = 0.5, 0.25, 0.125: %s | P = 0.0625 < T: %s | (2,4), f = -28: %s" % simnao(c))
ok = ok and all(c)

# --------------------------------------------- Rosenbrock
x0, a, P0, T = [-1.5, 2], 2, [0.5, 0.5], [1e-6, 1e-6]
print("\nRosenbrock a partir de (%g,%g), a = %g, P0 = (%g,%g), T = (%g,%g)" % (*x0, a, *P0, *T))
r = hooke_jeeves(rosen, x0, a, P0, T)
print(r.message)
hit = int(np.argmax(r.ftrace < 1e-4)) + 1
print("primeira avaliação com f < 1e-4: n_f = %d" % hit)
print("no fim: x = (%.6f, %.6f), f = %.2e;  %d movimentos, n_f total = %d"
      % (*r.x, r.fx, r.nit, r.nfev))
# custo de cada movimento: n a 2n (exploração), 1 + (n a 2n) (padrão), n = 2
H = r.history; n = 2
dn = np.diff(np.concatenate(([1], H[:, -1])))
ce = dn[H[:, 1] == 0]; cp = dn[H[:, 1] == 1]
print("custo por exploração: %d a %d; por movimento de padrão: %d a %d"
      % (ce.min(), ce.max(), cp.min(), cp.max()))
r5 = hooke_jeeves(rosen, x0, a, P0, [1e-5, 1e-5])
print("(com T = 1e-5: n_f = %d até f < 1e-4 e n_f total = %d)"
      % (int(np.argmax(r5.ftrace < 1e-4)) + 1, r5.nfev))
c = [hit == 353, bool(np.all((ce >= n) & (ce <= 2 * n))),
     bool(np.all((cp >= 1 + n) & (cp <= 1 + 2 * n)))]
print("  n_f = 353 até f < 1e-4: %s | n <= n_f <= 2n: %s | 1 + (n a 2n): %s" % simnao(c))
ok = ok and all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 3.2.5: método de Hooke–Jeeves

Exploração à mão: xb = (1,1), f(xb) = -16, P = (0.5,0.5)
  x1 + P1: (1.5, 1) -> f = -18.25, melhora: guarda-se
  x2 + P2: (1.5, 1.5) -> f = -21, melhora: guarda-se
xe = (1.5, 1.5), f = -21, com 2 avaliações
  f(1,1) = -16: sim | -18.25 e -21: sim | xe = (1.5,1.5): sim | 2 avaliações: sim

Exemplo completo: x0 = (1,1), a = 2, P0 = (0.5,0.5), T = (0.1,0.1)
   m movimento      xb1      xb2      xt1      xt2     f(xt)      xe1      xe2     f(xe)       P1       P2  resultado    n_f
   1 explor.     1.0000   1.0000        -        -         -   1.5000   1.5000  -21.0000   0.5000   0.5000  melhora        3
   2 padrão      1.0000   1.0000   2.0000   2.0000  -24.0000   2.0000   2.5000  -25.7500   0.5000   0.5000  aceite         7
   3 padrão      1.5000   1.5000   2.5000   3.5000  -27.0000   2.0000   4.0000  -28.0000   0.5000   0.5000  aceite        11
   4 padrão      2.0000   2.5000   2.0000   5.5000  -25.7500   2.0000   5.0000  -27.0000   0.5